# dtflowcv person/car GPU baseline

Upload `person_car_dataset.tar.gz` to Google Drive first, then run this notebook on a GPU runtime.

In [ ]:
!nvidia-smi
!pip install -q ultralytics mlflow

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# Change this if you upload the bundle somewhere else in Drive.
BUNDLE = '/content/drive/MyDrive/dtflowcv/person_car_dataset.tar.gz'
!rm -rf /content/person_car_dataset /content/runs /content/mlruns
!tar -xzf "$BUNDLE" -C /content
!find /content/person_car_dataset/images/val2017 -maxdepth 1 -type f | wc -l
!find /content/person_car_dataset/labels/val2017 -maxdepth 1 -type f | wc -l
!cat /content/person_car_dataset/dataset.yaml

In [ ]:
import csv
import json
import time
from pathlib import Path

import mlflow
import torch
from ultralytics import YOLO

DATASET = '/content/person_car_dataset/dataset.yaml'
RUN_DIR = Path('/content/runs/yolov8n-coco-person-car-gpu-100epoch')
NAMES = ['person', 'car']


def clean_key(key):
    return key.replace('metrics/', '').replace('(', '').replace(')', '').replace(':', '_').replace('/', '_')


def read_map50_curve(csv_path):
    if not csv_path.exists():
        return []
    with csv_path.open(newline='') as fh:
        rows = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(fh)]
    curve = []
    for row in rows:
        epoch = int(float(row.get('epoch', len(curve))))
        raw = row.get('metrics/mAP50(B)') or row.get('metrics/mAP50')
        if raw not in (None, ''):
            curve.append({'epoch': epoch, 'map50': float(raw)})
    return curve


def plateau_status(curve, window=10, min_delta=0.005):
    if len(curve) < window * 2:
        return {'status': 'insufficient_epochs', 'window': window, 'min_delta': min_delta}
    values = [point['map50'] for point in curve]
    best_before = max(values[:-window])
    best_recent = max(values[-window:])
    best_all = max(values)
    recent_gain = best_recent - best_before
    last_gap_to_best = best_all - values[-1]
    return {
        'status': 'plateau' if recent_gain < min_delta and last_gap_to_best < min_delta else 'not_plateau',
        'window': window,
        'min_delta': min_delta,
        'best_before_last_window': best_before,
        'best_recent': best_recent,
        'recent_gain': recent_gain,
        'best_all': best_all,
        'last_map50': values[-1],
        'last_gap_to_best': last_gap_to_best,
    }


mlflow.set_tracking_uri('file:/content/mlruns')
mlflow.set_experiment('dtflowcv-coco-person-car-baseline')
assert torch.cuda.is_available(), 'GPU runtime is required'

with mlflow.start_run(run_name='yolov8n-coco-person-car-gpu-100epoch') as run:
    model = YOLO('yolov8n.pt')
    started = time.perf_counter()
    result = model.train(
        data=DATASET,
        epochs=100,
        imgsz=640,
        batch=16,
        workers=8,
        device=0,
        seed=20260425,
        patience=30,
        project='/content/runs',
        name='yolov8n-coco-person-car-gpu-100epoch',
        exist_ok=True,
    )
    elapsed = time.perf_counter() - started
    train_metrics = {
        clean_key(key): float(value)
        for key, value in result.results_dict.items()
        if isinstance(value, (int, float))
    }
    train_metrics['train_elapsed_seconds'] = elapsed

    best_model = YOLO(str(RUN_DIR / 'weights' / 'best.pt'))
    val = best_model.val(
        data=DATASET,
        imgsz=640,
        batch=16,
        workers=8,
        device=0,
        split='val',
        plots=True,
        project='/content/runs',
        name='final_val',
        exist_ok=True,
    )
    box = val.box
    per_class_ap50 = {name: float(box.ap50[idx]) for idx, name in enumerate(NAMES)}
    per_class_ap50_95 = {name: float(box.ap[idx]) for idx, name in enumerate(NAMES)}
    final_metrics = {
        'map50': float(box.map50),
        'map50_95': float(box.map),
        'precision': float(box.mp),
        'recall': float(box.mr),
        'per_class_ap50': per_class_ap50,
        'per_class_ap50_95': per_class_ap50_95,
    }

    curve = read_map50_curve(RUN_DIR / 'results.csv')
    plateau = plateau_status(curve)
    summary = {
        'experiment_id': run.info.experiment_id,
        'run_id': run.info.run_id,
        'torch_cuda_name': torch.cuda.get_device_name(0),
        'dataset': DATASET,
        'run_dir': str(RUN_DIR),
        'best_weights': str(RUN_DIR / 'weights' / 'best.pt'),
        'train_metrics': train_metrics,
        'final_validation': final_metrics,
        'learning_curve_map50': curve,
        'plateau': plateau,
    }
    Path('/content/baseline_summary.json').write_text(
        json.dumps(summary, indent=2, sort_keys=True) + '\n',
        encoding='utf-8',
    )
    mlflow.log_metrics({key: value for key, value in train_metrics.items() if isinstance(value, (int, float))})
    mlflow.log_metric('final_map50', final_metrics['map50'])
    mlflow.log_metric('final_map50_95', final_metrics['map50_95'])
    mlflow.log_metric('final_person_ap50', per_class_ap50['person'])
    mlflow.log_metric('final_car_ap50', per_class_ap50['car'])
    mlflow.log_param('dataset', DATASET)
    mlflow.log_param('torch_cuda_name', torch.cuda.get_device_name(0))
    print(json.dumps(summary, indent=2, sort_keys=True))


In [ ]:
!tar -czf /content/person_car_gpu_results.tar.gz -C /content runs mlruns baseline_summary.json
!mkdir -p /content/drive/MyDrive/dtflowcv
!cp /content/person_car_gpu_results.tar.gz /content/drive/MyDrive/dtflowcv/person_car_gpu_results.tar.gz
!ls -lh /content/drive/MyDrive/dtflowcv/person_car_gpu_results.tar.gz
